In [ ]:
!pip install pandas selfies torch transformers deepchem rdkit swifter tqdm
!pip install faiss-cpu sentence-transformers pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.4/552.4 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.2/36.2 MB 58.7 MB/s eta 0:00:00
  Created wheel for swifter: filename=swifter-1.4.0-py3-none-any.whl size=16505 sha256=6f12efda6407e74a372c657be46d9106066bafa74e8f63140698ad9dea6a465b
  Stored in directory: /root/.cache/pip/wheels/d9/31/ff/ff51141a088571a9f672449e5aad5ea8bb35ca5d95ba135f30
Successfully built swifter
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 97.9 MB/s eta 0:00:00


In [ ]:
!pip install rank_bm25 openai

In [ ]:
"""
多索引RAG系统 - 彻底解决内存问题
策略：构建多个独立的小索引，避免单个索引过大
"""

import json
import pickle
import numpy as np
from pathlib import Path
from typing import List, Dict
import re
import os
import gc

import faiss
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# ============ 配置 ============
GDRIVE_BASE = "/content/drive/MyDrive"
PROJECT_NAME = "PatentRAG"
GDRIVE_PROJECT = f"{GDRIVE_BASE}/{PROJECT_NAME}"
os.makedirs(GDRIVE_PROJECT, exist_ok=True)

print(f"✓ 项目目录: {GDRIVE_PROJECT}")


# ============ 快速构建单个小索引 ============

def build_small_index(index_id: int, docs_per_index: int = 90):
    """
    构建单个小索引（安全！）

    Args:
        index_id: 索引编号 (0-9)
        docs_per_index: 每个索引包含的文档数
    """

    print(f"\n{'='*80}")
    print(f"🔨 构建索引 {index_id}")
    print(f"{'='*80}")

    # 1. 加载模型
    print("🔄 加载模型...")
    encoder = SentenceTransformer('all-MiniLM-L6-v2')
    dimension = 384

    # 2. 查找文档
    base_path = Path("/content/drive/MyDrive/MinerU")
    all_docs = sorted([f.parent for f in base_path.rglob("full.md")])

    start_idx = index_id * docs_per_index
    end_idx = min(start_idx + docs_per_index, len(all_docs))

    if start_idx >= len(all_docs):
        print(f"⚠️  索引 {index_id} 超出范围")
        return

    my_docs = all_docs[start_idx:end_idx]
    print(f"📝 处理文档 {start_idx+1}-{end_idx} ({len(my_docs)}个)")

    # 3. 创建索引
    index = faiss.IndexFlatIP(dimension)
    documents = []

    # 4. 处理文档
    for doc_dir in tqdm(my_docs, desc=f"索引{index_id}"):
        try:
            # 读取full.md
            full_md = (doc_dir / "full.md").read_text(encoding='utf-8')

            # 简单分块（按段落）
            chunks = [p.strip() for p in full_md.split('\n\n') if len(p.strip()) > 50]

            if not chunks:
                continue

            # 向量化（小批次）
            for i in range(0, len(chunks), 8):
                batch = chunks[i:i+8]
                embeddings = encoder.encode(batch, show_progress_bar=False, convert_to_numpy=True)
                embeddings = embeddings.astype('float32')
                faiss.normalize_L2(embeddings)

                index.add(embeddings)

                # 保存元数据
                for chunk in batch:
                    documents.append({
                        'content': chunk,
                        'metadata': {'doc_id': doc_dir.name, 'title': ''}
                    })
        except Exception as e:
            print(f"\n✗ {doc_dir.name[:30]}: {str(e)[:50]}")

    # 5. 保存这个小索引
    save_path = Path(f"{GDRIVE_PROJECT}/index_{index_id}")
    save_path.mkdir(parents=True, exist_ok=True)

    faiss.write_index(index, str(save_path / "faiss.index"))

    with open(save_path / "documents.pkl", 'wb') as f:
        pickle.dump({'documents': documents, 'dimension': dimension}, f)

    print(f"\n✅ 索引 {index_id} 完成: {len(documents)} 个文档块")
    print(f"💾 保存位置: {save_path}")

    # 清理内存
    del encoder, index, documents
    gc.collect()


# ============ 查询多个索引 ============

def search_all_indices(query: str, top_k: int = 5):
    """搜索所有小索引并合并结果"""

    print(f"🔍 搜索: {query}")

    # 加载模型
    encoder = SentenceTransformer('all-MiniLM-L6-v2')
    query_embedding = encoder.encode(query, convert_to_numpy=True)
    query_embedding = query_embedding.astype('float32').reshape(1, -1)
    faiss.normalize_L2(query_embedding)

    all_results = []

    # 搜索每个索引
    for i in range(10):  # 0-9共10个索引
        index_path = Path(f"{GDRIVE_PROJECT}/index_{i}")

        if not index_path.exists():
            continue

        # 加载索引
        index = faiss.read_index(str(index_path / "faiss.index"))
        with open(index_path / "documents.pkl", 'rb') as f:
            data = pickle.load(f)

        # 搜索
        distances, indices = index.search(query_embedding, top_k)

        for dist, idx in zip(distances[0], indices[0]):
            if idx < len(data['documents']):
                all_results.append({
                    'content': data['documents'][idx]['content'],
                    'metadata': data['documents'][idx]['metadata'],
                    'score': float(dist)
                })

    # 按分数排序
    all_results.sort(key=lambda x: x['score'], reverse=True)

    # 返回Top-K
    return all_results[:top_k]


# ============ 使用示例 ============

# 步骤1: 构建所有小索引（一次一个，避免内存问题）
for i in range(10):
    build_small_index(i, docs_per_index=90)

# 步骤2: 查询
results = search_all_indices("Which photoinitiators work better under LED?", top_k=5)

for i, r in enumerate(results, 1):
    print(f"\n[{i}] 分数: {r['score']:.3f}")
    print(f"内容: {r['content'][:200]}...")

✓ 项目目录: /content/drive/MyDrive/PatentRAG

🔨 构建索引 0
🔄 加载模型...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...


✅ 索引 7 完成: 9026 个文档块
💾 保存位置: /content/drive/MyDrive/PatentRAG/index_7

🔨 构建索引 8
🔄 加载模型...
📝 处理文档 721-810 (90个)


索引8: 100%|██████████| 90/90 [12:29<00:00,  8.33s/it]



✅ 索引 8 完成: 6103 个文档块
💾 保存位置: /content/drive/MyDrive/PatentRAG/index_8

🔨 构建索引 9
🔄 加载模型...
📝 处理文档 811-900 (90个)


索引9: 100%|██████████| 90/90 [19:28<00:00, 12.98s/it]



✅ 索引 9 完成: 9784 个文档块
💾 保存位置: /content/drive/MyDrive/PatentRAG/index_9
🔍 搜索: Which photoinitiators work better under LED?

[1] 分数: 0.750
内容: # Research Progress of Photolytic and LED Sensitive Photoinitiators...

[2] 分数: 0.648
内容: Abstract: At present, most of the photoinitiators used in LED photopolymerization systems were multicomponent photoinitiators, and only a few works involved single component photolytic photoinitiators...

[3] 分数: 0.639
内容: [12] C. Dietlin, S. Schweizer, P. Xiao, J. Zhang, F. Morlet- Savary, B. Graff, J. P. Fouassier, J. Lalevee, Photopolymerization upon LEDs: new photoinitiating systems and strategies, Polymer Chemistry...

[4] 分数: 0.638
内容: [0056] Suitable photoinitiators which can be used in combination with the thioxanthone photoinitiator of the present invention include, but are not limited to, those which are capable of absorbing UV ...

[5] 分数: 0.618
内容: Type I photoinitiating abilities of the studied compounds (0.5% w) were examined using RT- FTIR in t

In [ ]:
"""
多索引高级RAG系统
功能：多索引 + BM25混合检索 + 重排序 + DeepSeek
"""

import json
import pickle
import numpy as np
from pathlib import Path
from typing import List, Dict
import re

import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from openai import OpenAI

# ============ 配置 ============
GDRIVE_PROJECT = "/content/drive/MyDrive/PatentRAG"
NUM_INDICES = 10  # 索引数量


class MultiIndexRAG:
    """多索引RAG系统（支持高级功能）"""

    def __init__(self, use_reranker: bool = True, deepseek_api_key: str = None):
        """
        Args:
            use_reranker: 是否使用重排序
            deepseek_api_key: DeepSeek API密钥（可选）
        """
        print("🔄 初始化高级RAG系统...")

        # 向量检索模型
        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')

        # 重排序模型（可选）
        self.reranker = None
        if use_reranker:
            print("🔄 加载重排序模型...")
            self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
            print("✓ 重排序模型加载完成")

        # DeepSeek LLM（可选）
        self.llm = None
        if deepseek_api_key:
            self.llm = OpenAI(
                api_key=deepseek_api_key,
                base_url="https://api.deepseek.com"
            )
            print("✓ DeepSeek LLM已初始化")

        # BM25索引（延迟加载）
        self.bm25 = None
        self.bm25_docs = None

        print("✅ 系统初始化完成")

    def search_vector(self, query: str, top_k: int = 20) -> List[Dict]:
        """向量检索（多索引）"""

        # 向量化查询
        query_embedding = self.encoder.encode(query, convert_to_numpy=True)
        query_embedding = query_embedding.astype('float32').reshape(1, -1)
        faiss.normalize_L2(query_embedding)

        all_results = []

        # 搜索所有索引
        for i in range(NUM_INDICES):
            index_path = Path(f"{GDRIVE_PROJECT}/index_{i}")

            if not index_path.exists():
                continue

            # 加载索引
            index = faiss.read_index(str(index_path / "faiss.index"))
            with open(index_path / "documents.pkl", 'rb') as f:
                data = pickle.load(f)

            # 搜索
            distances, indices = index.search(query_embedding, top_k // NUM_INDICES + 5)

            for dist, idx in zip(distances[0], indices[0]):
                if idx < len(data['documents']):
                    all_results.append({
                        'content': data['documents'][idx]['content'],
                        'metadata': data['documents'][idx]['metadata'],
                        'score': float(dist),
                        'source': 'vector'
                    })

        # 按分数排序
        all_results.sort(key=lambda x: x['score'], reverse=True)

        return all_results[:top_k]

    def search_bm25(self, query: str, top_k: int = 20) -> List[Dict]:
        """BM25检索"""

        # 延迟构建BM25索引
        if self.bm25 is None:
            print("🔨 构建BM25索引（首次查询需要）...")
            self._build_bm25_index()

        # 分词
        query_tokens = re.findall(r'\b\w+\b', query.lower())

        # BM25评分
        scores = self.bm25.get_scores(query_tokens)

        # 获取Top-K
        top_indices = np.argsort(scores)[::-1][:top_k]

        results = []
        for idx in top_indices:
            if scores[idx] > 0:
                results.append({
                    'content': self.bm25_docs[idx]['content'],
                    'metadata': self.bm25_docs[idx]['metadata'],
                    'score': float(scores[idx]),
                    'source': 'bm25'
                })

        return results

    def _build_bm25_index(self):
        """构建BM25索引（从所有小索引中加载文档）"""

        all_docs = []
        tokenized_corpus = []

        for i in range(NUM_INDICES):
            index_path = Path(f"{GDRIVE_PROJECT}/index_{i}")
            if not index_path.exists():
                continue

            with open(index_path / "documents.pkl", 'rb') as f:
                data = pickle.load(f)

            for doc in data['documents']:
                all_docs.append(doc)
                tokens = re.findall(r'\b\w+\b', doc['content'].lower())
                tokenized_corpus.append(tokens)

        self.bm25_docs = all_docs
        self.bm25 = BM25Okapi(tokenized_corpus)

        print(f"✓ BM25索引完成: {len(all_docs):,} 个文档")

    def search_hybrid(
        self,
        query: str,
        top_k: int = 10,
        vector_weight: float = 0.7,
        use_rerank: bool = True
    ) -> List[Dict]:
        """
        混合检索（向量 + BM25 + 重排序）

        Args:
            query: 查询文本
            top_k: 返回结果数
            vector_weight: 向量检索权重（0-1）
            use_rerank: 是否使用重排序
        """

        # 1. 向量检索
        vector_results = self.search_vector(query, top_k=top_k * 2)

        # 2. BM25检索
        bm25_results = self.search_bm25(query, top_k=top_k * 2)

        # 3. 融合（归一化分数）
        def normalize_scores(results):
            if not results:
                return []
            scores = [r['score'] for r in results]
            min_s, max_s = min(scores), max(scores)
            range_s = max_s - min_s if max_s > min_s else 1.0

            for r in results:
                r['score'] = (r['score'] - min_s) / range_s
            return results

        vector_results = normalize_scores(vector_results)
        bm25_results = normalize_scores(bm25_results)

        # 合并（加权）
        merged = {}
        bm25_weight = 1.0 - vector_weight

        for r in vector_results:
            key = r['content'][:100]
            merged[key] = {
                'content': r['content'],
                'metadata': r['metadata'],
                'score': r['score'] * vector_weight,
                'source': 'hybrid'
            }

        for r in bm25_results:
            key = r['content'][:100]
            if key in merged:
                merged[key]['score'] += r['score'] * bm25_weight
            else:
                merged[key] = {
                    'content': r['content'],
                    'metadata': r['metadata'],
                    'score': r['score'] * bm25_weight,
                    'source': 'hybrid'
                }

        results = sorted(merged.values(), key=lambda x: x['score'], reverse=True)[:top_k * 2]

        # 4. 重排序（可选）
        if use_rerank and self.reranker is not None:
            pairs = [[query, r['content'][:500]] for r in results]
            rerank_scores = self.reranker.predict(pairs)

            for r, score in zip(results, rerank_scores):
                r['score'] = float(score)
                r['source'] = 'hybrid_reranked'

            results.sort(key=lambda x: x['score'], reverse=True)

        return results[:top_k]

    def query_with_llm(
        self,
        query: str,
        top_k: int = 5,
        use_hybrid: bool = True
    ) -> tuple:
        """
        使用LLM生成答案

        Returns:
            (answer, retrieved_results)
        """

        if self.llm is None:
            raise ValueError("请先设置 deepseek_api_key")

        # 检索
        if use_hybrid:
            results = self.search_hybrid(query, top_k=top_k)
        else:
            results = self.search_vector(query, top_k=top_k)

        # 构建上下文
        context = "\n\n".join([
            f"[文献 {i+1}] 相关度: {r['score']:.3f}\n"
            f"内容: {r['content'][:800]}"
            for i, r in enumerate(results[:5])
        ])

        # 构建Prompt
        prompt = f"""你是一位专业的化学文献分析助手。请基于以下文献片段回答用户的问题。

要求：
1. 直接回答问题，引用具体文献内容
2. 如果文献中没有直接答案，说明相关信息
3. 保持客观和专业

文献片段：
{context}

用户问题：{query}

回答："""

        # 调用DeepSeek
        response = self.llm.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": "你是一位专业的化学文献分析助手。"},
                {"role": "user", "content": prompt}
            ],
            max_tokens=2048,
            temperature=0.7
        )

        answer = response.choices[0].message.content

        return answer, results


# ============ 使用示例 ============

def example_usage():
    """完整使用示例"""

    print("="*80)
    print("🚀 多索引高级RAG系统")
    print("="*80)

    # 1. 初始化（不使用DeepSeek）
    rag = MultiIndexRAG(
        use_reranker=True,
        deepseek_api_key=None  # 暂不使用LLM
    )

    # 2. 测试不同检索策略
    query = "Which photoinitiators work better under LED light sources?"

    print(f"\n{'='*80}")
    print(f"查询: {query}")
    print(f"{'='*80}")

    # 2a. 纯向量检索
    print("\n【策略1】纯向量检索:")
    results = rag.search_vector(query, top_k=3)
    for i, r in enumerate(results, 1):
        print(f"[{i}] {r['score']:.3f} | {r['content'][:100]}...")

    # 2b. 混合检索 + 重排序
    print("\n【策略2】混合检索 + 重排序:")
    results = rag.search_hybrid(query, top_k=3)
    for i, r in enumerate(results, 1):
        print(f"[{i}] {r['score']:.3f} | {r['source']} | {r['content'][:100]}...")

    print("\n✅ 测试完成！")


def query_with_deepseek():
    """使用DeepSeek生成答案"""

    # 初始化（带DeepSeek）
    rag = MultiIndexRAG(
        use_reranker=True,
        deepseek_api_key="sk-xxxxxxxxxxxxx"  # 替换为你的API密钥
    )

    query = "Which photoinitiators work better under LED light sources?"

    # 生成答案
    answer, results = rag.query_with_llm(query, top_k=5, use_hybrid=True)

    print(f"\n💡 DeepSeek回答:")
    print("="*80)
    print(answer)
    print("="*80)

    print(f"\n📚 参考文献:")
    for i, r in enumerate(results, 1):
        print(f"[{i}] {r['score']:.3f} | {r['metadata']['doc_id'][:50]}")


# ============ 运行 ============
if __name__ == "__main__":
    # 先安装依赖
    # !pip install rank-bm25 openai -q

    # 运行示例
    # example_usage()
    query_with_deepseek()

# 📝 完整使用流程（在Colab中）

# # ============================================
# # Cell 1: 安装额外依赖
# # ============================================
# !pip install rank-bm25 openai -q

# # ============================================
# # Cell 2: 运行高级RAG（复制上面完整代码）
# # ============================================
# # （粘贴上面的完整代码）

# # ============================================
# # Cell 3: 测试（不使用DeepSeek）
# # ============================================
  # rag = MultiIndexRAG(use_reranker=True, deepseek_api_key=None)

  # # 混合检索测试
  # results = rag.search_hybrid(
  #     "Which photoinitiators work better under LED?",
  #     top_k=5
  # )

  # for i, r in enumerate(results, 1):
  #     print(f"\n[{i}] 分数: {r['score']:.3f} | 来源: {r['source']}")
  #     print(f"内容: {r['content'][:200]}...")

# # ============================================
# # Cell 4: 使用DeepSeek（可选）
# # ============================================
  # rag = MultiIndexRAG(
  #     use_reranker=True,
  #     deepseek_api_key="sk-xxxxxxxxxx"  # 填入你的API密钥
  # )

  # answer, refs = rag.query_with_llm(
  #     "如何选择适合LED光源的光引发剂？",
  #     top_k=5
  # )

  # print("💡 回答:")
  # print(answer)

🔄 初始化高级RAG系统...
🔄 加载重排序模型...
✓ 重排序模型加载完成
✓ DeepSeek LLM已初始化
✅ 系统初始化完成
🔨 构建BM25索引（首次查询需要）...
✓ BM25索引完成: 119,964 个文档

💡 DeepSeek回答:
根据文献内容，以下光引发剂在LED光源下表现更好：

1. **鎓盐类（onium salts）和肟酯类（oxime esters）光引发剂**（文献1）
   - 因其优异的光化学性能成为研究热点
   - 适用于LED光聚合体系

2. **三苯胺-六芳基双咪唑衍生物**（文献2）
   - 可作为氢受体光引发剂
   - 适用于紫外光和LED光源下的自由基光聚合

3. **酰基膦氧化物类**（文献3）
   - 包括但不限于：
     - 2,4,6-三甲基苯甲酰基-二苯基氧化膦
     - 2,4,6-三甲基苯甲酰基-二苯基膦酸盐
     - 双(2,4,6-三甲基苯甲酰基)苯基氧化膦
   - 能够吸收LED光源发射的特定波长（365-405nm）

4. **CMTX（特定硫杂蒽酮类）**（文献4）
   - 实验证明在LED固化下表现良好
   - 相比之下，水溶性光引发剂Irgacure 2959在LED下完全无法固化

需要注意的是，传统工业中常用的可裂解型光引发剂大多在LED发射下光吸收效率较差，需要红移吸收波长来适应LED光源（文献1）。目前开发对LED敏感的新型光引发剂是该领域的迫切需求。

📚 参考文献:
[1] 6.801 | 光可裂解型LED敏感的光引发剂的研究进展_谢洁.pdf-f55e60c8-69ea-4263-9ef
[2] 4.564 | A review of oxime-ester V6_HAL.pdf-81552d8d-4c22-4
[3] 4.476 | WO2015183719A1.pdf-1875eb29-e833-401a-b735-5e92b28
[4] 3.557 | WO2015183719A1.pdf-1875eb29-e833-401a-b735-5e92b28
[5] 2.460 | 光可裂解型LED敏感的光引发剂的研究进展_谢洁.pdf-f55e60c8-69ea-4263-9ef


In [ ]:
# -*- coding: utf-8 -*-
"""
多模态RAG系统 - 基于CLIP
支持：文本检索 + 图片检索 + 图文混合检索
"""

import json
import pickle
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple
import re
import os
import gc
from PIL import Image

import faiss
import torch
from sentence_transformers import SentenceTransformer
from transformers import CLIPProcessor, CLIPModel
from tqdm import tqdm

# ============ 配置 ============
GDRIVE_BASE = "/content/drive/MyDrive"
PROJECT_NAME = "MultimodalRAG"
GDRIVE_PROJECT = f"{GDRIVE_BASE}/{PROJECT_NAME}"
os.makedirs(GDRIVE_PROJECT, exist_ok=True)

print(f"✓ 项目目录: {GDRIVE_PROJECT}")


class MultimodalIndexBuilder:
    """多模态索引构建器（文本 + 图片）"""

    def __init__(self, clip_model_name: str = "openai/clip-vit-base-patch32"):
        """
        Args:
            clip_model_name: CLIP模型名称
        """
        print("🔄 加载CLIP模型...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.clip_model = CLIPModel.from_pretrained(clip_model_name).to(self.device)
        self.clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
        self.dimension = 512  # CLIP embedding维度
        print(f"✓ CLIP模型加载完成 (设备: {self.device})")

    def encode_text(self, texts: List[str]) -> np.ndarray:
        """
        用CLIP编码文本

        Args:
            texts: 文本列表

        Returns:
            embeddings: (N, 512) numpy数组
        """
        inputs = self.clip_processor(text=texts, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            text_features = self.clip_model.get_text_features(**inputs)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)  # L2归一化

        return text_features.cpu().numpy()

    def encode_image(self, image_paths: List[str]) -> np.ndarray:
        """
        用CLIP编码图片

        Args:
            image_paths: 图片路径列表

        Returns:
            embeddings: (N, 512) numpy数组
        """
        images = []
        for path in image_paths:
            try:
                img = Image.open(path).convert('RGB')
                images.append(img)
            except Exception as e:
                print(f"✗ 无法加载图片 {path}: {e}")
                # 创建黑色占位图
                images.append(Image.new('RGB', (224, 224)))

        inputs = self.clip_processor(images=images, return_tensors="pt", padding=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            image_features = self.clip_model.get_image_features(**inputs)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        return image_features.cpu().numpy()

    def build_multimodal_index(
        self,
        index_id: int,
        docs_per_index: int = 50,
        base_path: str = "/content/drive/MyDrive/MinerU"
    ):
        """
        构建多模态索引（文本 + 图片统一索引）

        Args:
            index_id: 索引编号
            docs_per_index: 每个索引包含的文档数
            base_path: MinerU输出目录
        """
        print(f"\n{'='*80}")
        print(f"🔨 构建多模态索引 {index_id}")
        print(f"{'='*80}")

        # 查找文档
        base_path = Path(base_path)
        all_docs = sorted([f.parent for f in base_path.rglob("full.md")])

        start_idx = index_id * docs_per_index
        end_idx = min(start_idx + docs_per_index, len(all_docs))

        if start_idx >= len(all_docs):
            print(f"⚠️  索引 {index_id} 超出范围")
            return

        my_docs = all_docs[start_idx:end_idx]
        print(f"📝 处理文档 {start_idx+1}-{end_idx} ({len(my_docs)}个)")

        # 创建索引（文本和图片共用一个索引）
        index = faiss.IndexFlatIP(self.dimension)
        documents = []  # 存储文本块和图片的元数据

        # 处理每个文档
        for doc_dir in tqdm(my_docs, desc=f"索引{index_id}"):
            try:
                # 1. 处理文本
                full_md = (doc_dir / "full.md").read_text(encoding='utf-8')
                text_chunks = [p.strip() for p in full_md.split('\n\n') if len(p.strip()) > 50]

                # 提取图片引用和上下文
                image_contexts = self._extract_image_contexts(full_md, doc_dir)

                # 2. 文本向量化（批量处理）
                if text_chunks:
                    for i in range(0, len(text_chunks), 8):
                        batch = text_chunks[i:i+8]
                        embeddings = self.encode_text(batch)
                        embeddings = embeddings.astype('float32')

                        index.add(embeddings)

                        for chunk in batch:
                            documents.append({
                                'type': 'text',
                                'content': chunk,
                                'metadata': {'doc_id': doc_dir.name, 'title': ''}
                            })

                # 3. 图片向量化
                if image_contexts:
                    image_paths = [ctx['image_path'] for ctx in image_contexts]
                    valid_paths = [p for p in image_paths if Path(p).exists()]

                    if valid_paths:
                        # 批量编码图片
                        for i in range(0, len(valid_paths), 4):  # 图片批量小一些
                            batch_paths = valid_paths[i:i+4]
                            batch_contexts = [ctx for ctx in image_contexts
                                            if ctx['image_path'] in batch_paths]

                            embeddings = self.encode_image(batch_paths)
                            embeddings = embeddings.astype('float32')

                            index.add(embeddings)

                            for ctx in batch_contexts:
                                documents.append({
                                    'type': 'image',
                                    'content': ctx['context'],
                                    'image_path': ctx['image_path'],
                                    'caption': ctx['caption'],
                                    'metadata': {'doc_id': doc_dir.name, 'title': ''}
                                })

            except Exception as e:
                print(f"\n✗ {doc_dir.name[:30]}: {str(e)[:50]}")

        # 保存索引
        save_path = Path(f"{GDRIVE_PROJECT}/index_{index_id}")
        save_path.mkdir(parents=True, exist_ok=True)

        faiss.write_index(index, str(save_path / "multimodal.index"))

        with open(save_path / "documents.pkl", 'wb') as f:
            pickle.dump({'documents': documents, 'dimension': self.dimension}, f)

        text_count = sum(1 for d in documents if d['type'] == 'text')
        image_count = sum(1 for d in documents if d['type'] == 'image')

        print(f"\n✅ 索引 {index_id} 完成:")
        print(f"   - 文本块: {text_count}")
        print(f"   - 图片: {image_count}")
        print(f"   - 总计: {len(documents)}")
        print(f"💾 保存位置: {save_path}")

        # 清理内存
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def _extract_image_contexts(self, markdown: str, doc_dir: Path) -> List[Dict]:
        """
        从Markdown中提取图片及其上下文

        Args:
            markdown: markdown文本
            doc_dir: 文档目录

        Returns:
            图片上下文列表: [{'image_path': ..., 'caption': ..., 'context': ...}]
        """
        results = []

        # 正则匹配图片: ![caption](images/xxx.png)
        pattern = r'!\[(.*?)\]\((images/[^\)]+)\)'
        matches = re.finditer(pattern, markdown)

        for match in matches:
            caption = match.group(1)
            image_rel_path = match.group(2)
            image_path = doc_dir / image_rel_path

            if not image_path.exists():
                continue

            # 提取图片周围的文本作为上下文（前后各200字符）
            pos = match.start()
            context_start = max(0, pos - 200)
            context_end = min(len(markdown), pos + 200)
            context = markdown[context_start:context_end].strip()

            results.append({
                'image_path': str(image_path),
                'caption': caption,
                'context': context
            })

        return results


class MultimodalRAG:
    """多模态RAG检索系统"""

    def __init__(self, clip_model_name: str = "openai/clip-vit-base-patch32"):
        print("🔄 初始化多模态RAG系统...")

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.clip_model = CLIPModel.from_pretrained(clip_model_name).to(self.device)
        self.clip_processor = CLIPProcessor.from_pretrained(clip_model_name)

        print("✅ 系统初始化完成")

    def search(
        self,
        query: str = None,
        query_image: str = None,
        top_k: int = 10,
        filter_type: str = None  # 'text', 'image', None
    ) -> List[Dict]:
        """
        多模态检索（支持文本查询、图片查询、或混合查询）

        Args:
            query: 文本查询（可选）
            query_image: 图片路径（可选）
            top_k: 返回结果数
            filter_type: 过滤结果类型（'text'只返回文本，'image'只返回图片）

        Returns:
            检索结果列表
        """
        if query is None and query_image is None:
            raise ValueError("必须提供文本查询或图片查询")

        # 1. 编码查询
        if query and query_image:
            # 混合查询：图文各占50%
            text_emb = self._encode_text([query])
            image_emb = self._encode_image([query_image])
            query_embedding = (text_emb + image_emb) / 2
        elif query:
            query_embedding = self._encode_text([query])
        else:
            query_embedding = self._encode_image([query_image])

        query_embedding = query_embedding.astype('float32')

        # 2. 搜索所有索引
        all_results = []

        index_dirs = sorted(Path(GDRIVE_PROJECT).glob("index_*"))
        for index_path in index_dirs:
            if not (index_path / "multimodal.index").exists():
                continue

            # 加载索引
            index = faiss.read_index(str(index_path / "multimodal.index"))
            with open(index_path / "documents.pkl", 'rb') as f:
                data = pickle.load(f)

            # 搜索
            distances, indices = index.search(query_embedding, top_k * 2)

            for dist, idx in zip(distances[0], indices[0]):
                if idx < len(data['documents']):
                    doc = data['documents'][idx]

                    # 类型过滤
                    if filter_type and doc['type'] != filter_type:
                        continue

                    all_results.append({
                        **doc,
                        'score': float(dist)
                    })

        # 3. 排序并返回Top-K
        all_results.sort(key=lambda x: x['score'], reverse=True)
        return all_results[:top_k]

    def _encode_text(self, texts: List[str]) -> np.ndarray:
        """编码文本"""
        inputs = self.clip_processor(text=texts, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            features = self.clip_model.get_text_features(**inputs)
            features = features / features.norm(dim=-1, keepdim=True)

        return features.cpu().numpy()

    def _encode_image(self, image_paths: List[str]) -> np.ndarray:
        """编码图片"""
        images = [Image.open(p).convert('RGB') for p in image_paths]
        inputs = self.clip_processor(images=images, return_tensors="pt", padding=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            features = self.clip_model.get_image_features(**inputs)
            features = features / features.norm(dim=-1, keepdim=True)

        return features.cpu().numpy()

    def display_results(self, results: List[Dict]):
        """美化显示检索结果"""
        for i, r in enumerate(results, 1):
            print(f"\n{'='*80}")
            print(f"[{i}] 类型: {r['type'].upper()} | 相关度: {r['score']:.3f}")
            print(f"文档ID: {r['metadata']['doc_id'][:50]}")

            if r['type'] == 'text':
                print(f"内容预览: {r['content'][:200]}...")
            else:
                print(f"图片路径: {r['image_path']}")
                print(f"图片说明: {r['caption']}")
                print(f"上下文: {r['content'][:150]}...")


# ============ 使用示例 ============

def build_all_indices():
    """构建所有多模态索引"""
    builder = MultimodalIndexBuilder()

    # 假设有1000个文档，分成20个索引（每个50个文档）
    for i in range(20):
        builder.build_multimodal_index(i, docs_per_index=50)

        # 清理内存
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def example_text_query():
    """示例1: 纯文本查询（可以检索到文本和图片）"""
    rag = MultimodalRAG()

    query = "LED photoinitiators absorption spectrum"
    print(f"\n🔍 文本查询: {query}")

    results = rag.search(query=query, top_k=5)
    rag.display_results(results)


def example_image_query():
    """示例2: 以图搜图/文"""
    rag = MultimodalRAG()

    query_image = "/path/to/query_image.png"
    print(f"\n🖼️  图片查询: {query_image}")

    results = rag.search(query_image=query_image, top_k=5)
    rag.display_results(results)


def example_hybrid_query():
    """示例3: 图文混合查询"""
    rag = MultimodalRAG()

    query = "absorption spectrum"
    query_image = "/path/to/spectrum.png"

    print(f"\n🔍🖼️  混合查询:")
    print(f"   文本: {query}")
    print(f"   图片: {query_image}")

    results = rag.search(query=query, query_image=query_image, top_k=5)
    rag.display_results(results)


def example_filter_results():
    """示例4: 只返回图片或只返回文本"""
    rag = MultimodalRAG()

    query = "chemical structure"

    # 只返回图片
    print(f"\n🖼️  只检索图片:")
    image_results = rag.search(query=query, top_k=5, filter_type='image')
    rag.display_results(image_results)

    # 只返回文本
    print(f"\n📝 只检索文本:")
    text_results = rag.search(query=query, top_k=5, filter_type='text')
    rag.display_results(text_results)


# ============ 运行 ============
if __name__ == "__main__":
    # 安装依赖
    # !pip install transformers torch pillow faiss-cpu -q

    # 步骤1: 构建索引（只需运行一次）
    build_all_indices()

    # 步骤2: 检索测试
    # example_text_query()
    # example_image_query()
    # example_hybrid_query()
    # example_filter_results()

    pass


✓ 项目目录: /content/drive/MyDrive/MultimodalRAG
🔄 加载CLIP模型...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

索引2: 100%|██████████| 50/50 [10:51<00:00, 13.02s/it]



✅ 索引 2 完成:
   - 文本块: 6301
   - 图片: 790
   - 总计: 7091
💾 保存位置: /content/drive/MyDrive/MultimodalRAG/index_2

🔨 构建多模态索引 3
📝 处理文档 151-200 (50个)


索引3: 100%|██████████| 50/50 [07:49<00:00,  9.39s/it]



✅ 索引 3 完成:
   - 文本块: 4830
   - 图片: 561
   - 总计: 5391
💾 保存位置: /content/drive/MyDrive/MultimodalRAG/index_3

🔨 构建多模态索引 4
📝 处理文档 201-250 (50个)


索引4:  86%|████████▌ | 43/50 [15:15<00:39,  5.61s/it]